# Harmonize scored-event labels of an .edf EEG/PSG database

This notebook lets you **visualize** the scored-event configurations present in a heterogeneous
.edf database (events annotated during sleep scoring and exported by Profusion/Compumedics:
apnea, hypopnea, arousals, limb movements, PLM, SpO2 desaturation…) and **harmonize** their raw
labels to a single canonical vocabulary.

It returns a JSON file `config_param/event_remap.json` (a flat python dict
`{raw_label: canonical_label}`, with `null` for labels you choose to ignore) that downstream tools
(epoch rejection, event-epoch viewer) can use.

---
**To use this notebook, interact with the widgets and read the output below. You first have to
select a database to make the widgets appear.**

Sections:
1. Select your study folder and scan the events
1bis. *(optional)* Check that the text/CSV export and the `*.edf.XML` agree
2. Event configurations found
3. Harmonize the labels
4. Preview & save the JSON
5. Verify

The events are read, in priority order, from the `*_ScoredEvents_Export.txt` text export (Compumedics/
Curry French export), then the `*_event_xml.csv`, then the `<ScoredEvents>` of the `*.edf.XML` (Profusion
`CMPStudyConfig`).

---
last update 19/06/2026, YN


In [ ]:
# Imports
try:
    import os
    import re
    import json
    import datetime
    import traceback
    import unicodedata
    import xml.etree.ElementTree as ET
    from pathlib import Path
    from collections import Counter, OrderedDict
    import pandas as pd
    import ipywidgets as widgets
    from ipyfilechooser import FileChooser
    from IPython.display import display, HTML, clear_output
except ImportError as e:
    print("⚠️ Error: ", e)
else:
    print("✅ Packages and functions successfully imported!")


# ---- generic helpers (shared idioms with 2_select&remap_channels_edf) ----
def _load_json_lenient(path):
    """Load a JSON file, tolerating a single trailing comma before a closing } or ]
    (a common hand-edit mistake): strict parse first, repair only on failure."""
    with open(path, encoding="utf-8") as f:
        text = f.read()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return json.loads(re.sub(r",(\s*[}\]])", r"\1", text))


def print_in_scrollable_box(text, height=300, font_size="12px"):
    display(HTML(f'<pre style="overflow-y:scroll; height:{height}px; border:1px solid black; '
                 f'padding:10px; font-size:{font_size};">{text}</pre>'))


# ---- canonical event vocabulary (editable suggestions only) ----
# Compumedics/Profusion raw labels -> harmonized snake_case canonical labels.
# Arousal subtypes are kept (clinically meaningful); laterality is dropped for limbs.
DEFAULT_EVENT_MAPPING = {
    "obstructive apnea":   "apnea_obstructive",
    "central apnea":       "apnea_central",
    "mixed apnea":         "apnea_mixed",
    "hypopnea":            "hypopnea",
    "spo2 desaturation":   "spo2_desaturation",
    "spo2 artifact":       "spo2_artifact",
    "arousal (aro res)":   "arousal_respiratory",
    "arousal (aro spont)": "arousal_spontaneous",
    "arousal (aro plm)":   "arousal_limb",
    "arousal (aro limb)":  "arousal_limb",
    "arousal ()":          "arousal",
    "arousal":             "arousal",
    "limb movement (left)":  "limb_movement",
    "limb movement (right)": "limb_movement",
    "limb movement":         "limb_movement",
    "plm (left)":  "plm",
    "plm (right)": "plm",
    "plm":         "plm",
    "snore":       "snore",
    "snoring":     "snore",
}
# ---- French text-export labels (*_ScoredEvents_Export.txt) ----
# The French export writes variable label text ("Micro-éveil 2 ARO RES"), so match on
# informative substrings (accent-insensitive) rather than exact keys.
FRENCH_EVENT_RULES = [
    ("aro spont",         "arousal_spontaneous"),
    ("aro res",           "arousal_respiratory"),
    ("aro plm",           "arousal_limb"),
    ("aro limb",          "arousal_limb"),
    ("apnee obstructive", "apnea_obstructive"),
    ("apnee centrale",    "apnea_central"),
    ("apnee mixte",       "apnea_mixed"),
    ("hypopnee",          "hypopnea"),
    ("desaturation",      "spo2_desaturation"),
    ("artefact spo2",     "spo2_artifact"),
    ("ronflement",        "snore"),
    ("plm",               "plm"),
]
CANONICAL_VOCAB = sorted(set(DEFAULT_EVENT_MAPPING.values())
                         | {canon for _, canon in FRENCH_EVENT_RULES})


def _strip_accents(text):
    """Lower-case and drop accents so French labels match regardless of accentuation
    (é→e, É→e). Used for the substring rules above."""
    return "".join(c for c in unicodedata.normalize("NFKD", text.lower())
                   if not unicodedata.combining(c))


def suggest_canonical(raw):
    """Suggest a canonical label for a raw event name (empty string if unknown).
    English labels (Compumedics CSV/XML) match an exact dict; French labels (the
    *_ScoredEvents_Export.txt export) match accent-insensitive substrings."""
    key = raw.strip().lower()
    if key in DEFAULT_EVENT_MAPPING:
        return DEFAULT_EVENT_MAPPING[key]
    # tolerate a trailing laterality marker, e.g. "Limb Movement (Left)"
    stripped = re.sub(r"\s*\((left|right)\)\s*$", "", key).strip()
    if stripped in DEFAULT_EVENT_MAPPING:
        return DEFAULT_EVENT_MAPPING[stripped]
    # French text export: informative-substring match (accent-insensitive)
    norm = _strip_accents(raw)
    for sub, canon in FRENCH_EVENT_RULES:
        if sub in norm:
            return canon
    return ""


# ---- EDF recording start (only needed to convert text-export clock times to seconds) ----
def read_edf_start_datetime(edf_path):
    """Read the EDF recording-start datetime from the fixed header (offset 168 = date
    'dd.mm.yy', 176 = time 'hh.mm.ss'), applying the EDF 2-digit-year clipping
    (00-84 -> 20xx, 85-99 -> 19xx). Header-only read; returns a datetime or None on failure."""
    try:
        with open(edf_path, "rb") as f:
            f.seek(168)
            date_str = f.read(8).decode("ascii", "replace").strip()   # dd.mm.yy
            time_str = f.read(8).decode("ascii", "replace").strip()   # hh.mm.ss
        dd, mm, yy = (int(x) for x in date_str.split("."))
        hh, mi, ss = (int(x) for x in time_str.split("."))
        year = 2000 + yy if yy <= 84 else 1900 + yy
        return datetime.datetime(year, mm, dd, hh, mi, ss)
    except Exception:
        return None


# ---- event companion loading: TXT first, then CSV, then XML (<ScoredEvents>) fallback ----
def event_companion_paths(edf_path, txt_suffix="_ScoredEvents_Export.txt",
                          csv_suffix="_event_xml.csv"):
    """Return (txt_path_or_None, csv_path_or_None, xml_path_or_None) for an EDF stem.
    txt_suffix / csv_suffix are the configurable Compumedics event-export suffixes."""
    edf_path = Path(edf_path)
    txt = edf_path.with_name(f"{edf_path.stem}{txt_suffix}")
    txt = txt if txt.exists() else None
    csv = edf_path.with_name(f"{edf_path.stem}{csv_suffix}")
    csv = csv if csv.exists() else None
    xml = None
    for cand in (f"{edf_path.name}.XML", f"{edf_path.name}.xml"):
        p = edf_path.with_name(cand)
        if p.exists():
            xml = p
            break
    return txt, csv, xml


def load_events_from_csv(csv_path):
    """Parse a Compumedics *_event_xml.csv -> list of (name, start, duration)."""
    df = pd.read_csv(csv_path)
    for col in ("Name", "Start", "Duration"):
        if col not in df.columns:
            raise ValueError(f"missing column '{col}' in {Path(csv_path).name}")
    events = []
    for _, row in df.iterrows():
        events.append((str(row["Name"]).strip(), float(row["Start"]), float(row["Duration"])))
    return events


def load_events_from_xml(xml_path):
    """Parse the <ScoredEvents> of a Profusion CMPStudyConfig .edf.XML
    -> list of (name, start, duration). <Input> is ignored (absent from the CSV)."""
    root = ET.parse(xml_path).getroot()
    events = []
    for se in root.iter("ScoredEvent"):
        name_el = se.find("Name")
        if name_el is None or name_el.text is None:
            continue
        start_el = se.find("Start")
        dur_el = se.find("Duration")
        start = float(start_el.text) if (start_el is not None and start_el.text) else float("nan")
        dur = float(dur_el.text) if (dur_el is not None and dur_el.text) else float("nan")
        events.append((name_el.text.strip(), start, dur))
    return events


_TXT_DUR_RE = re.compile(r"^(\d+):(\d+(?:\.\d+)?)$")   # "M:SS" or "M:SS.s"


def load_events_from_txt(txt_path, rec_start=None):
    """Parse a Compumedics/Curry French text export (*_ScoredEvents_Export.txt)
    -> list of (name, start, duration). Comma-separated, no header, columns:
        HH:MM:SS , epoch# , stage_FR , event_label_FR , M:SS[.s] , - , - , position
    Encoding varies (UTF-16 with BOM on some exports, UTF-8/ANSI on others) -> the BOM is
    sniffed. Clock times are converted to seconds-from-recording-start (with midnight
    rollover) when rec_start is given; without it (harmonization only needs the names)
    Start and Duration are returned as NaN, so no EDF-header read is required for the scan."""
    raw = open(txt_path, "rb").read()
    if raw[:2] in (b"\xff\xfe", b"\xfe\xff"):
        text = raw.decode("utf-16")
    else:
        text = raw.decode("utf-8", errors="replace")
    rec_date = rec_start.date() if rec_start is not None else None
    events = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        parts = [p.strip() for p in line.split(",")]
        if len(parts) < 5:
            continue
        name = parts[3]
        if rec_start is None:
            events.append((name, float("nan"), float("nan")))
            continue
        # clock time -> seconds from recording start (hours may be single-digit)
        try:
            hh, mm, ss = parts[0].split(":")
            clock_t = datetime.time(int(hh), int(mm), int(ss))
        except (ValueError, TypeError):
            events.append((name, float("nan"), float("nan")))
            continue
        event_dt = datetime.datetime.combine(rec_date, clock_t)
        # events recorded after midnight fall on the next calendar day
        if (rec_start - event_dt).total_seconds() > 3600:
            event_dt += datetime.timedelta(days=1)
        start_sec = (event_dt - rec_start).total_seconds()
        m = _TXT_DUR_RE.match(parts[4])
        dur_sec = int(m.group(1)) * 60 + float(m.group(2)) if m else 0.0
        events.append((name, start_sec, dur_sec))
    return events


def load_events(edf_path, txt_suffix="_ScoredEvents_Export.txt", csv_suffix="_event_xml.csv"):
    """TXT-first, then CSV, then XML-fallback event loader.
    Returns (events_list, source) with source in {'txt', 'csv', 'xml'} or (None, None) when
    no companion is usable. events_list = list of (name, start, duration). For the TXT source
    only the names are needed here (harmonization), so Start/Duration are left NaN (no EDF
    header read); the 1bis check reads them with the recording-start datetime."""
    txt, csv, xml = event_companion_paths(edf_path, txt_suffix, csv_suffix)
    if txt is not None:
        try:
            return load_events_from_txt(txt), "txt"
        except Exception:
            pass  # fall through to the CSV
    if csv is not None:
        try:
            return load_events_from_csv(csv), "csv"
        except Exception:
            pass  # fall through to the XML
    if xml is not None:
        try:
            return load_events_from_xml(xml), "xml"
        except Exception:
            pass
    return None, None


def load_existing_mapping(folder):
    """Read the existing config_param/event_remap.json (lenient), {} if absent/unreadable."""
    p = Path(folder) / "config_param" / "event_remap.json"
    if p.exists():
        try:
            return _load_json_lenient(p)
        except Exception:
            return {}
    return {}


def _canon_events(events):
    """List of (canonical_label, floor(start_seconds)) for cross-source comparison.
    Names are normalized to the canonical vocabulary (suggest_canonical, raw-lowercase fallback)
    so English (CSV/XML) and French (.txt) labels compare equal; start times are floored to whole
    seconds because the .txt export truncates clock times to the second."""
    return [(suggest_canonical(n) or n.strip().lower(), int(s)) for n, s, _d in events]


def _match_events(primary, xml, tol):
    """Greedy match of primary events to XML events within +/- tol whole seconds.
    primary / xml: lists of (canonical_label, floor_second) from _canon_events.
    Returns (n_matched, cooccur_pairs, only_in_primary, only_in_xml):
      - pass 1 pairs events with the SAME canonical label within tol (nearest second wins);
      - pass 2 pairs any leftover events within tol -> a co-occurring pair with DIFFERENT labels,
        recorded in cooccur_pairs as a {(primary_label, xml_label): count} Counter. Inspect those
        pairs to lift the ambiguity between "same event, unharmonized name" (e.g. a cross-language
        pair worth mapping together) and "two genuinely distinct events that happen to fall within
        tol seconds" (unrelated labels);
      - anything still unpaired is only-in-one-source.
    tol=0 reduces to strict same-second matching. Greedy O(n*m) — fine for a few hundred events."""
    P, X = list(primary), list(xml)
    used_x = [False] * len(X)
    matched_p = [False] * len(P)
    n_matched = 0
    cooccur = Counter()

    def _nearest(ps, pl, same_label):
        best_j, best_d = -1, tol + 1
        for j, (xl, xs) in enumerate(X):
            if used_x[j] or (same_label and xl != pl):
                continue
            d = abs(ps - xs)
            if d <= tol and d < best_d:
                best_j, best_d = j, d
        return best_j

    for i, (pl, ps) in enumerate(P):                 # pass 1: same label within tol
        j = _nearest(ps, pl, True)
        if j >= 0:
            used_x[j] = matched_p[i] = True
            n_matched += 1
    for i, (pl, ps) in enumerate(P):                 # pass 2: any label within tol -> co-occurrence
        if matched_p[i]:
            continue
        j = _nearest(ps, pl, False)
        if j >= 0:
            used_x[j] = matched_p[i] = True
            cooccur[(pl, X[j][0])] += 1
    only_p = Counter(pl for i, (pl, _s) in enumerate(P) if not matched_p[i])
    only_x = Counter(xl for j, (xl, _s) in enumerate(X) if not used_x[j])
    return n_matched, cooccur, only_p, only_x


# ---- shared state filled by the scan ----
STATE = {}


# ========================= Section banners =========================
section1 = widgets.HTML("""
<hr style="height:4px; background-color:black; border:none;">
<h2>1. Select your study folder and scan the events</h2>
<p>Pick the folder of your .edf database. Each .edf is expected to have a Compumedics/Profusion
event companion next to it. Three sources are supported, in priority order:
<br>&#x2022; the <b>text export</b> <code>*_ScoredEvents_Export.txt</code> (read first; suffix in
the <b>TXT suffix</b> field — default <code>_ScoredEvents_Export.txt</code>);
<br>&#x2022; then the <b>CSV</b> <code>*_event_xml.csv</code> (suffix in the <b>CSV suffix</b> field);
<br>&#x2022; then, as a fallback, the <code>&lt;ScoredEvents&gt;</code> of the <code>*.edf.XML</code>.
<br>&#x2022; Selecting the folder auto-detects both suffixes and refreshes the info lines below.
<br>&#x2022; Click <b>Run scan</b> to read the events and list the configurations.
<br>&#x2022; "Skip labels already mapped" hides labels already present in an existing
<code>event_remap.json</code> (incremental harmonization when you add a new cohort).</p>
""")

section1bis = widgets.HTML("""
<hr style="height:4px; background-color:black; border:none;">
<h2>1bis. (Optional) Check text/CSV vs XML consistency</h2>
<p>For every file having <b>both</b> a primary source (the <code>.txt</code> text export if present,
else the <code>*_event_xml.csv</code>) <b>and</b> the <code>&lt;ScoredEvents&gt;</code> of the
<code>*.edf.XML</code>, checks that the two describe the same events. Labels are normalized to the
<b>canonical vocabulary</b> (so English XML and French <code>.txt</code> compare equal) and events are
matched by <b>type + start time within ±(Match&nbsp;tolerance) seconds</b> (default 1&nbsp;s — the
<code>.txt</code> truncates clock times to the second and the two exports can round a start
differently; set 0 for strict same-second matching). Events paired within the tolerance but carrying
<b>different labels</b> are listed in <code>cooccur_label_pairs</code> as candidate same-events whose
names are not yet harmonized (e.g. a cross-language pair) — <b>inspect those pairs to decide whether
they are truly one event or two distinct events that merely fall within the tolerance</b>. Events with
no counterpart show up as only-in-one-source (e.g. an export that omits snoring). Writes
<code>config_param/event_source_mismatch.tsv</code>. Opt-in because it reads both files per EDF.</p>
""")

section2 = widgets.HTML("""
<hr style="height:4px; background-color:black; border:none;">
<h2>2. Event configurations found</h2>
<p>Files are grouped by their set of <b>unique</b> event labels. Two files with the same unique
labels share one configuration even if their event counts differ.</p>
""")

section3 = widgets.HTML("""
<hr style="height:4px; background-color:black; border:none;">
<h2>3. Harmonize the labels</h2>
<p>For each unique raw label, choose a harmonized (canonical) label. Defaults are pre-filled
when recognized; you can edit them freely.
<br>&#x2022; Tick <b>ignore</b> to drop a label (stored as <code>null</code>, excluded downstream).
<br>&#x2022; Click <b>Validate mapping &amp; ignores</b> to check your choices and unlock section 4.
<br>&#x2022; Then go to section 4 to preview &amp; save.</p>
""")

section4 = widgets.HTML("""
<hr style="height:4px; background-color:black; border:none;">
<h2>4. Preview &amp; save the JSON</h2>
<p>Builds a flat <code>{raw_label: canonical_label}</code> mapping and <b>merges</b> it into any
existing <code>config_param/event_remap.json</code> (labels mapped this session replace their old
value, all others are kept).</p>
""")

section5 = widgets.HTML("""
<hr style="height:4px; background-color:black; border:none;">
<h2>5. Verify</h2>
<p>Applies the saved mapping to every configuration and reports the resulting harmonized labels.
The verdict passes when no raw label is left unmapped (ignored labels count as handled).</p>
""")

# ========================= Section 1 widgets =========================
chooser = FileChooser(os.getcwd())
chooser.title = "<b>Choose your study folder</b>"
chooser.show_only_dirs = True

skip_existing = widgets.Checkbox(value=True, description="Skip labels already mapped",
                                 style={"description_width": "initial"})
existing_info = widgets.HTML(value="")
txt_suffix = widgets.Text(value="_ScoredEvents_Export.txt", description="TXT suffix:",
                          style={"description_width": "initial"},
                          layout=widgets.Layout(width="420px"))
txt_suffix_info = widgets.HTML(value="")
csv_suffix = widgets.Text(value="_event_xml.csv", description="CSV suffix:",
                          style={"description_width": "initial"},
                          layout=widgets.Layout(width="420px"))
csv_suffix_info = widgets.HTML(value="")
run_scan_button = widgets.Button(description="Run scan", button_style="success", icon="play")
out_scan = widgets.Output()

# Section 1bis
run_check_button = widgets.Button(description="Run text/CSV vs XML check", button_style="info", icon="check")
tol_seconds = widgets.BoundedIntText(value=1, min=0, max=10, description="Match tolerance (s):",
                                     style={"description_width": "initial"},
                                     layout=widgets.Layout(width="170px"))
out_check = widgets.Output()

# Section 2 / 3 / 4 / 5 output zones
out_configs = widgets.Output()
out_harmonize = widgets.Output()
fname_text = widgets.Text(value="event_remap.json", description="File name:",
                          style={"description_width": "initial"}, layout=widgets.Layout(width="320px"))
preview_save_button = widgets.Button(description="Preview & Save", button_style="success",
                                     icon="save", disabled=True)
out_save = widgets.Output()
validate_button = widgets.Button(description="Validate mapping & ignores",
                                 button_style="primary", icon="check")
out_validate = widgets.Output()
verify_scope = widgets.Dropdown(options=["All configurations", "Only configs with unmapped labels"],
                                value="All configurations", description="Show:",
                                style={"description_width": "initial"}, layout=widgets.Layout(width="380px"))
verify_button = widgets.Button(description="Run verification", button_style="success", icon="play")
out_verify = widgets.Output()


# ========================= Callbacks =========================
def _update_info(*_):
    """Refresh the info line when the folder changes (no scan)."""
    if not getattr(chooser, "selected_path", None):
        existing_info.value = ""
        csv_suffix_info.value = ""
        return
    try:
        folder = Path(chooser.selected_path)
        edfs = [f for f in folder.rglob("*") if f.suffix.lower() == ".edf" and not f.name.startswith("._")]
        if not edfs:
            existing_info.value = '<small style="color:#888;">No EDF files found in selected folder.</small>'
            csv_suffix_info.value = ""
            txt_suffix_info.value = ""
            return
        existing = load_existing_mapping(folder)
        msg = f"<small>{len(edfs)} EDF file(s) found. "
        msg += (f"{len(existing)} label(s) already mapped in event_remap.json."
                if existing else "No existing event_remap.json yet.")
        existing_info.value = msg + "</small>"
        # --- Event TXT-export suffix auto-detection (Compumedics/Curry *_ScoredEvents_Export.txt) ---
        # Scan .txt files whose name contains 'event' so hypnogram .txt files are not counted.
        all_txt = [f for f in folder.rglob("*")
                   if f.suffix.lower() == ".txt" and "event" in f.name.lower()]
        txt_counts = {}
        for edf in edfs:
            for tf in all_txt:
                if os.path.normcase(tf.name).startswith(os.path.normcase(edf.stem)):
                    suf = tf.name[len(edf.stem):]
                    txt_counts[suf] = txt_counts.get(suf, 0) + 1
        if not txt_counts:
            txt_suffix_info.value = (
                '<small style="color:#e67e00;">No event .txt detected next to the EDFs '
                '— set the suffix manually or rely on CSV/XML.</small>')
        else:
            best_suffix, best_count = max(
                txt_counts.items(), key=lambda x: (x[1], -len(x[0])))
            txt_suffix.value = best_suffix
            parts = [f'<b>{s}</b>&nbsp;(×{c})'
                     for s, c in sorted(txt_counts.items(), key=lambda x: -x[1])]
            color = '#2e7d32' if best_count == len(edfs) else '#e67e00'
            txt_suffix_info.value = (
                f'<small style="color:{color};">Detected:&nbsp;'
                f'{"&nbsp;·&nbsp;".join(parts)}&nbsp;— '
                f'{best_count}/{len(edfs)} EDF file(s) matched</small>')
        # --- Event-CSV suffix auto-detection (mirrors 5_quality_overview hypno-suffix block) ---
        all_csv = [f for f in folder.rglob("*") if f.suffix.lower() == ".csv"]
        suffix_counts = {}
        for edf in edfs:
            for csvf in all_csv:
                if os.path.normcase(csvf.name).startswith(os.path.normcase(edf.stem)):
                    suf = csvf.name[len(edf.stem):]
                    suffix_counts[suf] = suffix_counts.get(suf, 0) + 1
        if not suffix_counts:
            csv_suffix_info.value = (
                '<small style="color:#e67e00;">No event CSV detected next to the EDFs '
                '— set the suffix manually or rely on the XML fallback.</small>')
        else:
            # Events have no "more specific remapped" variant (unlike hypnograms), so prefer the
            # MOST FREQUENT suffix (shortest on ties).
            best_suffix, best_count = max(
                suffix_counts.items(), key=lambda x: (x[1], -len(x[0])))
            csv_suffix.value = best_suffix
            parts = [f'<b>{s}</b>&nbsp;(×{c})'
                     for s, c in sorted(suffix_counts.items(), key=lambda x: -x[1])]
            color = '#2e7d32' if best_count == len(edfs) else '#e67e00'
            csv_suffix_info.value = (
                f'<small style="color:{color};">Detected:&nbsp;'
                f'{"&nbsp;·&nbsp;".join(parts)}&nbsp;— '
                f'{best_count}/{len(edfs)} EDF file(s) matched</small>')
    except Exception as e:
        existing_info.value = f'<small style="color:#c33;">{type(e).__name__}: {e}</small>'
        csv_suffix_info.value = ""


def run_scan(_=None):
    for z in (out_scan, out_configs, out_harmonize, out_validate, out_save, out_verify):
        z.clear_output()
    with out_scan:
        if not getattr(chooser, "selected_path", None):
            print("⚠️ Please select your study folder first.")
            return
        folder = Path(chooser.selected_path)
        edfs = sorted(f for f in folder.rglob("*")
                      if f.suffix.lower() == ".edf" and not f.name.startswith("._"))
        if not edfs:
            print("⚠️ No EDF files found in the selected folder.")
            return
        configs = {}          # frozenset(labels) -> [file_id]
        label_files = {}      # raw label -> set(file_id)
        label_counts = {}     # raw label -> total occurrences
        source_by_file = {}   # file_id -> 'txt'/'csv'/'xml'
        failed = []           # (file_id, reason)
        n_with = 0
        for edf in edfs:
            fid = edf.stem
            try:
                events, source = load_events(edf, txt_suffix.value, csv_suffix.value)
            except Exception as e:
                failed.append((fid, f"{type(e).__name__}: {e}"))
                continue
            if events is None:
                failed.append((fid, "no readable event companion (.txt, .csv or .edf.XML)"))
                continue
            n_with += 1
            source_by_file[fid] = source
            names = [n for (n, _, _) in events]
            configs.setdefault(frozenset(names), []).append(fid)
            for nm in names:
                label_counts[nm] = label_counts.get(nm, 0) + 1
            for nm in set(names):
                label_files.setdefault(nm, set()).add(fid)
        STATE.update(folder=folder, configs=configs, label_files=label_files,
                     label_counts=label_counts, source_by_file=source_by_file,
                     failed=failed, existing=load_existing_mapping(folder))
        if failed:
            cfgdir = folder / "config_param"
            cfgdir.mkdir(exist_ok=True)
            pd.DataFrame(failed, columns=["file_id", "reason"]).to_csv(
                cfgdir / "failed_event_read.tsv", sep="\t", index=False)
        print(f"✅ Scanned {len(edfs)} EDF file(s): {n_with} with events, {len(failed)} failed.")
        print(f"   {len(label_files)} unique raw label(s), {len(configs)} distinct event configuration(s).")
        n_txt = sum(1 for s in source_by_file.values() if s == "txt")
        n_csv = sum(1 for s in source_by_file.values() if s == "csv")
        n_xml = sum(1 for s in source_by_file.values() if s == "xml")
        print(f"   event source: {n_txt} from TXT, {n_csv} from CSV, {n_xml} from XML fallback.")
        if STATE["existing"]:
            print(f"   {len(STATE['existing'])} label(s) already in event_remap.json.")
        if failed:
            print(f"   ⚠ {len(failed)} file(s) without readable events "
                  f"— see config_param/failed_event_read.tsv")
    render_configs()
    render_harmonize()


def render_configs():
    with out_configs:
        clear_output()
        configs = STATE.get("configs", {})
        if not configs:
            return
        items = sorted(configs.items(), key=lambda kv: (-len(kv[1]), sorted(kv[0])))

        # ---- per-config viewer: a dropdown selects one configuration to detail ----
        dd_options = [(f"Configuration {i} — {len(fids)} file(s), {len(labels)} unique label(s)", i - 1)
                      for i, (labels, fids) in enumerate(items, 1)]
        config_dd = widgets.Dropdown(options=dd_options, value=0, description="Show config:",
                                     style={"description_width": "initial"},
                                     layout=widgets.Layout(width="520px"))
        show_ids_btn = widgets.ToggleButton(value=False, description="Show file ids",
                                            icon="list-ul", layout=widgets.Layout(width="170px"))
        detail_out = widgets.Output()
        ids_out = widgets.Output()

        def _render_detail(*_):
            labels, fids = items[config_dd.value]
            labs = sorted(labels)
            with detail_out:
                clear_output()
                display(HTML(f"<b>{len(labs)} unique label(s)</b> in {len(fids)} file(s):<br>"
                             + "<br>".join(f"&#x2022; {l}" for l in labs)))
            show_ids_btn.description = "Hide file ids" if show_ids_btn.value else "Show file ids"
            with ids_out:
                clear_output()
                if show_ids_btn.value:
                    display(HTML('<pre style="overflow:auto; max-height:140px; border:1px solid #ccc; '
                                 'padding:6px; font-size:12px; margin-top:4px;">'
                                 + ", ".join(sorted(fids)) + "</pre>"))

        config_dd.observe(_render_detail, names="value")
        show_ids_btn.observe(_render_detail, names="value")
        display(widgets.VBox([config_dd, detail_out, show_ids_btn, ids_out]))
        _render_detail()

        # ---- global table of all unique raw labels (kept) ----
        lf, lc = STATE["label_files"], STATE["label_counts"]
        rows = sorted(lf.keys(), key=lambda l: (-len(lf[l]), l.lower()))
        df = pd.DataFrame([{"raw_label": l, "n_files": len(lf[l]), "n_occurrences": lc[l],
                            "suggested_canonical": suggest_canonical(l) or "(none)"} for l in rows])
        display(HTML("<h4>All unique raw labels</h4>"))
        display(HTML(df.to_html(index=False)))


def render_harmonize(*_):
    with out_harmonize:
        clear_output()
        out_validate.clear_output()
        preview_save_button.disabled = True   # require a fresh validation after any (re)build
        lf = STATE.get("label_files", {})
        if not lf:
            return
        existing = STATE.get("existing", {})
        labels = sorted(lf.keys(), key=lambda l: (-len(lf[l]), l.lower()))
        if skip_existing.value:
            labels = [l for l in labels if l not in existing]
        STATE["rows"] = {}
        row_widgets = []
        if not labels:
            display(HTML("<i>All raw labels are already mapped (uncheck “Skip labels "
                         "already mapped” to edit them).</i>"))
            return
        header = widgets.HBox([
            widgets.HTML("<b>raw label</b> (files)", layout=widgets.Layout(width="320px")),
            widgets.HTML("&rarr; <b>canonical label</b>", layout=widgets.Layout(width="240px")),
            widgets.HTML("<b>ignore</b>", layout=widgets.Layout(width="90px")),
        ])
        for raw in labels:
            if raw in existing:
                val = existing[raw]
                combo_val, ignore_val = ("", True) if val is None else (val, False)
            else:
                combo_val, ignore_val = suggest_canonical(raw), False
            lab = widgets.HTML(f"<code>{raw}</code> <small style='color:#888'>({len(lf[raw])})</small>",
                               layout=widgets.Layout(width="320px"))
            combo = widgets.Combobox(value=combo_val, options=CANONICAL_VOCAB, ensure_option=False,
                                     placeholder="canonical label", disabled=ignore_val,
                                     layout=widgets.Layout(width="240px"))
            ign = widgets.Checkbox(value=ignore_val, description="ignore", indent=False,
                                   layout=widgets.Layout(width="90px"))

            def _toggle(change, c=combo):
                c.disabled = change["new"]
            ign.observe(_toggle, names="value")
            combo.observe(_invalidate_save, names="value")
            ign.observe(_invalidate_save, names="value")
            STATE["rows"][raw] = (combo, ign)
            row_widgets.append(widgets.HBox([lab, combo, ign], layout=widgets.Layout(
                border="1px solid #e0e0e0", padding="4px", margin="0 0 2px 0",
                align_items="center", overflow="hidden")))
        box = widgets.VBox(row_widgets, layout=widgets.Layout(
            max_height="420px", overflow="auto", border="1px solid #ddd", padding="6px"))
        display(widgets.VBox([header, box]))


def _invalidate_save(*_):
    """Any change in section 3 invalidates a prior validation: the saved JSON must
    always reflect the latest selection, so re-lock section 4 until re-validated."""
    if not preview_save_button.disabled:
        preview_save_button.disabled = True
        out_save.clear_output()
        with out_validate:
            clear_output()
            display(HTML("<i style='color:#c98a00'>⚠ Selection changed — click "
                         "<b>Validate mapping &amp; ignores</b> again before saving.</i>"))


def on_validate(_=None):
    with out_validate:
        clear_output()
        rows = STATE.get("rows", {})
        if not rows:
            print("⚠️ Run the scan (section 1) first.")
            return
        mapped, ignored, empties = [], [], []
        for raw, (combo, ign) in rows.items():
            if ign.value:
                ignored.append(raw)
            elif combo.value.strip():
                mapped.append(raw)
            else:
                empties.append(raw)
        display(HTML(f"<b>Validation summary:</b> {len(mapped)} mapped, "
                     f"{len(ignored)} ignored, {len(empties)} left empty."))
        if empties:
            display(HTML("<span style='color:#c33'>⚠ empty (will NOT be saved): "
                         + ", ".join(f"<code>{e}</code>" for e in empties)
                         + " — set a canonical label or tick “ignore”.</span>"))
        display(HTML("<span style='color:#178a17'>✅ Section 4 unlocked — you can now "
                     "<b>Preview &amp; Save</b> below.</span>"))
        preview_save_button.disabled = False


def on_preview_save(_=None):
    with out_save:
        clear_output()
        rows = STATE.get("rows", {})
        if not rows:
            print("⚠️ Run the scan (section 1) first.")
            return
        try:
            session, empties = {}, []
            for raw, (combo, ign) in rows.items():
                if ign.value:
                    session[raw] = None
                else:
                    v = combo.value.strip()
                    if v:
                        session[raw] = v
                    else:
                        empties.append(raw)
            folder = STATE["folder"]
            existing = load_existing_mapping(folder)   # reload fresh to merge
            merged = dict(existing)
            merged.update(session)
            merged = OrderedDict(sorted(merged.items(), key=lambda kv: kv[0].lower()))
            cfgdir = folder / "config_param"
            cfgdir.mkdir(exist_ok=True)
            out_json = cfgdir / (fname_text.value.strip() or "event_remap.json")
            with open(out_json, "w", encoding="utf-8") as f:
                json.dump(merged, f, indent=2, ensure_ascii=False)
            n_new = len(session)
            display(HTML(f"<p><b>✅ JSON saved here:</b> <code>{out_json}</code></p>"))
            display(HTML(f"<p style='color:#555'>{len(merged)} label(s) total "
                         f"(<b>{n_new}</b> added/updated this session); previous entries preserved.</p>"))
            if empties:
                display(HTML("<p style='color:#c33'>⚠ left unmapped (not saved): "
                             + ", ".join(f"<code>{e}</code>" for e in empties)
                             + " — set a canonical label or tick “ignore”.</p>"))
            preview = json.dumps(merged, indent=2, ensure_ascii=False)
            print_in_scrollable_box(preview.replace("<", "&lt;").replace(">", "&gt;"), height=260)
            display(HTML("<br><b>To load this file later:</b>"))
            display(HTML("<p><code>with open(path, encoding='utf-8') as f: event_map = json.load(f)</code></p>"))
            display(HTML("<p><code>canonical = event_map.get(raw_label)  # None = ignore</code></p>"))
        except Exception as e:
            display(HTML(f"<pre style='color:#c33'>{type(e).__name__}: {e}\n"
                         f"{traceback.format_exc()}</pre>"))


def run_verify(_=None):
    with out_verify:
        clear_output()
        configs = STATE.get("configs", {})
        if not configs:
            print("⚠️ Run the scan (section 1) first.")
            return
        try:
            folder = STATE["folder"]
            mapping = load_existing_mapping(folder)
            if not mapping:
                print("⚠️ No event_remap.json found yet — save it in section 4 first.")
                return
            items = sorted(configs.items(), key=lambda kv: (-len(kv[1]), sorted(kv[0])))
            any_unmapped = False
            html = []
            for i, (labels, fids) in enumerate(items, 1):
                labs = sorted(labels)
                unmapped = [l for l in labs if l not in mapping]
                canon = sorted({mapping[l] for l in labs if mapping.get(l) is not None})
                if unmapped:
                    any_unmapped = True
                if verify_scope.value.startswith("Only") and not unmapped:
                    continue
                html.append(f"<h4>Configuration {i} &mdash; {len(fids)} file(s)</h4>")
                html.append("<b>Harmonized labels:</b> "
                            + (", ".join(f"<code>{c}</code>" for c in canon) or "<i>none</i>"))
                if unmapped:
                    html.append("<br><span style='color:#c33'><b>Unmapped:</b> "
                                + ", ".join(f"<code>{u}</code>" for u in unmapped) + "</span>")
                html.append("<hr>")
            if not html:
                html.append("<i>Nothing to show for this scope.</i>")
            display(HTML("".join(html)))
            if any_unmapped:
                display(HTML("<p style='color:#c33'><b>❌ Some raw labels are still unmapped.</b> "
                             "Map them in section 3 and save again.</p>"))
            else:
                display(HTML("<p style='color:#178a17'><b>✅ All raw labels are mapped "
                             "(or explicitly ignored).</b></p>"))
        except Exception as e:
            display(HTML(f"<pre style='color:#c33'>{type(e).__name__}: {e}</pre>"))


def run_events_consistency_check(_=None):
    with out_check:
        clear_output()
        if not getattr(chooser, "selected_path", None):
            print("⚠️ Please select your study folder first.")
            return
        folder = Path(chooser.selected_path)
        edfs = sorted(f for f in folder.rglob("*")
                      if f.suffix.lower() == ".edf" and not f.name.startswith("._"))
        if not edfs:
            print("⚠️ No EDF files found.")
            return

        def _summary(counter):
            # per-canonical-type counts, e.g. "snore×426; hypopnea×2"
            return "; ".join(f"{lbl}×{n}" for lbl, n in counter.most_common())

        rows = []
        for edf in edfs:
            fid = edf.stem
            txt, csv, xml = event_companion_paths(edf, txt_suffix.value, csv_suffix.value)
            # Primary text source = the .txt if present, else the CSV; compared against the XML.
            primary = "txt" if txt is not None else ("csv" if csv is not None else "")
            blank = dict(file_id=fid, primary_source=primary, match_tol_s=tol_seconds.value,
                         n_primary="", n_xml="",
                         n_matched="", n_cooccur_difflabel="",
                         only_in_primary="", only_in_xml="", cooccur_label_pairs="")
            if primary == "" or xml is None:
                rows.append({**blank,
                             "status": "missing text/CSV" if primary == "" else "missing XML"})
                continue
            try:
                if primary == "txt":
                    rec_start = read_edf_start_datetime(edf)
                    if rec_start is None:
                        rows.append({**blank, "status": "cannot read EDF start datetime"})
                        continue
                    pev = _canon_events(load_events_from_txt(txt, rec_start))
                else:
                    pev = _canon_events(load_events_from_csv(csv))
                xev = _canon_events(load_events_from_xml(xml))
            except Exception as e:
                rows.append({**blank, "status": f"error: {type(e).__name__}"})
                continue

            # Match events across sources within +/- tol seconds (see _match_events).
            n_matched, cooccur, only_p, only_x = _match_events(pev, xev, tol_seconds.value)
            n_cooccur = sum(cooccur.values())
            n_only_p, n_only_x = sum(only_p.values()), sum(only_x.values())
            status = "match" if (n_only_p == 0 and n_only_x == 0 and n_cooccur == 0) else "MISMATCH"
            pair_str = "; ".join(f"{pl}↔{xl}×{n}" for (pl, xl), n in cooccur.most_common(10))
            rows.append(dict(
                file_id=fid, primary_source=primary, match_tol_s=tol_seconds.value,
                n_primary=len(pev), n_xml=len(xev),
                n_matched=n_matched, n_cooccur_difflabel=n_cooccur,
                only_in_primary=_summary(only_p), only_in_xml=_summary(only_x),
                cooccur_label_pairs=pair_str, status=status))
        df = pd.DataFrame(rows)
        cfgdir = folder / "config_param"
        cfgdir.mkdir(exist_ok=True)
        df.to_csv(cfgdir / "event_source_mismatch.tsv", sep="\t", index=False)
        n_match = int((df["status"] == "match").sum())
        n_mis = int((df["status"] == "MISMATCH").sum())
        n_other = len(df) - n_match - n_mis
        color = "#178a17" if (n_mis == 0 and n_other == 0) else "#c98a00"
        display(HTML(f"<p style='color:{color}'><b>{n_match} match</b>, {n_mis} mismatch, "
                     f"{n_other} missing-companion/error — of {len(df)} file(s).</p>"))
        display(HTML(f"<p style='color:#555'>Events matched by <b>canonical label + start time within "
                     f"±{tol_seconds.value}s</b>. <code>cooccur_label_pairs</code> lists events paired in "
                     "time but with <b>different labels</b> — likely the same event with an unharmonized "
                     "(e.g. cross-language) name; if the paired labels are unrelated they may instead be "
                     "two distinct events falling within the tolerance.</p>"))
        display(HTML(f"<p style='color:#555'>Saved <code>{cfgdir / 'event_source_mismatch.tsv'}</code></p>"))
        show = df[df["status"] != "match"]
        if len(show):
            display(HTML("<b>Files needing attention:</b>"))
            display(HTML(show.to_html(index=False)))


# ========================= Wiring & layout =========================
chooser.register_callback(_update_info)
skip_existing.observe(lambda ch: render_harmonize(), names="value")
run_scan_button.on_click(run_scan)
run_check_button.on_click(run_events_consistency_check)
preview_save_button.on_click(on_preview_save)
validate_button.on_click(on_validate)
verify_button.on_click(run_verify)

ui_layout = widgets.VBox([
    section1, chooser, txt_suffix, txt_suffix_info, csv_suffix, csv_suffix_info, existing_info, skip_existing, run_scan_button, out_scan,
    section1bis, widgets.HBox([run_check_button, tol_seconds]), out_check,
    section2, out_configs,
    section3, out_harmonize, validate_button, out_validate,
    section4, widgets.HBox([fname_text, preview_save_button]), out_save,
    section5, widgets.HBox([verify_scope, verify_button]), out_verify,
])
display(ui_layout)
